# Laboratorio 03: Scraping CityPeru - Reseñas de Restaurantes

Caso de uso: Reseñas de restaurantes

Nota: Por cada restaurante se extraen metadatos (nombre, enlace, calificación, dirección) y reseñas (calificación, texto, usuario)

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import random
import re
import pandas as pd

In [2]:
URL_LISTADO = 'https://cityperu.com/restaurante/departamento/lima/'

TOTAL_RESEÑAS = 1000
MAX_RESTAURANTES = 34
RESEÑAS_POR_RESTAURANTE = 100

# Tiempos de espera aleatorios entre peticiones (en segundos)
DELAY_MIN = 5
DELAY_MAX = 10

# Reintentos consecutivos antes de una pausa larga
REINTENTOS_MAX = 5

In [3]:
# Configuración del navegador Chrome
options = Options()
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('--lang=es')

# Inicialización del driver y espera explícita
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20) 

In [4]:
# Obtiene los enlaces de los restaurantes que aparecen primero en el listado de la pagina
def obtener_restaurantes(driver, url_listado, max_restaurantes):
    driver.get(url_listado)
    time.sleep(5)
    
    # Busca todos los enlaces que contienen '/restaurante/'
    enlaces = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/restaurante/"]')
    
    urls_vistas = set()
    restaurantes = []
    
    for enlace in enlaces:
        href = enlace.get_attribute('href')
        if href and href not in urls_vistas:
            # Excluir el enlace del listado mismo
            if href.rstrip('/') != url_listado.rstrip('/'):
                urls_vistas.add(href)
                restaurantes.append(href)
                if len(restaurantes) >= max_restaurantes:
                    break
    
    return restaurantes

In [5]:
# Extrae nombre, calificación y dirección de un restaurante
def extraer_metadatos_restaurante(driver, url):
    # Nombre del restaurante desde el título mediante la etiqueta HTML h1
    try:
        nombre = driver.find_element(By.CSS_SELECTOR, 'h1').text.strip()
    except:
        nombre = None
    
    # Calificación general desde las clases de estrellas
    try:
        stars_elem = driver.find_element(By.CSS_SELECTOR, 'div.stars-review')
        clases = stars_elem.get_attribute('class')
        match = re.search(r'stars-review-(\d)-(\d)', clases)
        calificacion = float(f'{match.group(1)}.{match.group(2)}') if match else None
    except:
        calificacion = None
    
    # Dirección del restaurante
    try:
        direccion = driver.find_element(By.CSS_SELECTOR, 'div.address-ficha').text.strip()
    except:
        direccion = None
    
    return {
        'nombre_restaurante': nombre,
        'enlace': url,
        'calificacion_general': calificacion,
        'direccion_restaurante': direccion
    }

In [6]:
# Extrae las reseñas de la página actual
def extraer_reseñas(driver, pagina):
    reseñas = []
    # Tarjetas de reseña
    tarjetas = driver.find_elements(By.CSS_SELECTOR, 'div[class*="pb3"][class*="pt3"][class*="bt"]')
    
    for tarjeta in tarjetas:
        try:
            # Usuario que escribió la reseña
            try:
                usuario = tarjeta.find_element(By.CSS_SELECTOR, 'span.ml3').text.strip()
            except:
                usuario = None
            
            # Calificación de la reseña desde las estrellas
            try:
                stars_elem = tarjeta.find_element(By.CSS_SELECTOR, 'div.stars-review')
                clases = stars_elem.get_attribute('class')
                match = re.search(r'stars-review-(\d)-(\d)', clases)
                calificacion = float(f'{match.group(1)}.{match.group(2)}') if match else None
            except:
                calificacion = None
            
            # Texto de la reseña (el span más largo)
            try:
                spans = tarjeta.find_elements(By.CSS_SELECTOR, 'div.w-100 > span')
                texto = max((s.text.strip() for s in spans if len(s.text.strip()) > 20), key=len, default=None)
            except:
                texto = None
            
            # Solo guardar si hay texto válido
            if texto and len(texto) > 10:
                reseñas.append({
                    'calificacion_resena': calificacion,
                    'texto': texto,
                    'usuario': usuario,
                    'pagina': pagina
                })
        except:
            continue
    
    return reseñas

In [7]:
# Obtener la lista de restaurantes automáticamente del listado
restaurantes = obtener_restaurantes(driver, URL_LISTADO, MAX_RESTAURANTES)
print(f'Se encontraron {len(restaurantes)} restaurantes en el listado')

datos = []
errores_consecutivos = 0

# Recorrer cada restaurante y extraer sus reseñas
for idx, url_restaurante in enumerate(restaurantes):
    # Se detiene en caso de que ya se haya alcanzado el total de reseñas
    if len(datos) >= TOTAL_RESEÑAS:
        break
    
    print(f'\nRestaurante {idx + 1}/{len(restaurantes)}: {url_restaurante}')
    
    try:
        # Cargar la página de reseñas del restaurante
        driver.get(url_restaurante + '?view_reviews=yy#reviews')
        time.sleep(5)
        
        # Esperar a que carguen las tarjetas de reseñas
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[class*="pb3"][class*="pt3"][class*="bt"]')))
        time.sleep(3)
        
        # Extraer metadatos del restaurante
        metadatos = extraer_metadatos_restaurante(driver, url_restaurante)
        print(f'  Nombre: {metadatos["nombre_restaurante"]}')
        print(f'  Calificación: {metadatos["calificacion_general"]}')
        
        reseñas_restaurante = []
        pagina = 1
        ids_vistos = set()
        
        # Paginar hasta alcanzar el límite por restaurante
        while len(reseñas_restaurante) < RESEÑAS_POR_RESTAURANTE:
            reseñas_pagina = extraer_reseñas(driver, pagina)
            
            # En caso de que no hay reseñas, se termina el bucle
            if not reseñas_pagina:
                break
            
            # Filtrar duplicados por los primeros 50 caracteres del texto
            nuevas = [r for r in reseñas_pagina if r['texto'][:50] not in ids_vistos]
            for r in nuevas:
                ids_vistos.add(r['texto'][:50])
            
            # Si no hay nuevas reseñas, se termina el bucle
            if not nuevas:
                break
            
            reseñas_restaurante.extend(nuevas)
            print(f'  Página {pagina}: {len(nuevas)} nuevas | Total restaurante: {len(reseñas_restaurante)}')
            
            pagina += 1
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
            
            # Desplazar hacia abajo para cargar más contenido
            driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
            time.sleep(3)
        
        # Agregar metadatos a cada reseña
        for r in reseñas_restaurante:
            r.update(metadatos)
        
        datos.extend(reseñas_restaurante)
        errores_consecutivos = 0
        print(f'  Total restaurante: {len(reseñas_restaurante)} | Total general: {len(datos)}')
        
        # Guardar un punto de guardado por cada 50 registros (reseñas)
        if len(datos) % 50 < len(reseñas_restaurante):
            df_temp = pd.DataFrame(datos)
            df_temp.to_csv('reseñas_restaurantes.csv', index=False, encoding='utf-8-sig')
            print(f'  Checkpoint guardado: {len(datos)} reseñas')
        
        time.sleep(random.uniform(8, 12))
        
    except Exception as e:
        errores_consecutivos += 1
        print(f'  Error en restaurante: {e}')
        if errores_consecutivos >= REINTENTOS_MAX:
            print('  Demasiados errores. Pausa de 60s.')
            time.sleep(60)
            errores_consecutivos = 0
        else:
            time.sleep(10)

driver.quit()
print(f'\nScraping finalizado. Total: {len(datos)} reseñas')

Se encontraron 34 restaurantes en el listado

Restaurante 1/34: https://cityperu.com/restaurante/
  Error en restaurante: Message: 


Restaurante 2/34: https://cityperu.com/restaurante/lince/siete-sopas/
  Nombre: Opiniones de Siete Sopas Lince. (Restaurante) en Lince (Lima).
  Calificación: 4.3
  Página 1: 103 nuevas | Total restaurante: 103
  Total restaurante: 103 | Total general: 103
  Checkpoint guardado: 103 reseñas

Restaurante 3/34: https://cityperu.com/restaurante/miraflores/panchita-miraflores/
  Nombre: Opiniones de Panchita - Miraflores. (Restaurante) en Miraflores (Lima).
  Calificación: 4.6
  Página 1: 95 nuevas | Total restaurante: 95
  Total restaurante: 95 | Total general: 198
  Checkpoint guardado: 198 reseñas

Restaurante 4/34: https://cityperu.com/restaurante/miraflores/punto-azul-miraflores/
  Nombre: Opiniones de Punto Azul. (Restaurante) en Miraflores (Lima).
  Calificación: 4.6
  Página 1: 89 nuevas | Total restaurante: 89
  Total restaurante: 89 | Total general

In [8]:
# Almacenar reseñas en un archivo CSV
df = pd.DataFrame(datos)
df = df.drop_duplicates(subset=['texto'])
df.to_csv('reseñas_restaurantes.csv', index=False, encoding='utf-8-sig')
print(f'DataFrame final: {len(df)} reseñas únicas')
print(f'\nRestaurantes únicos: {df["nombre_restaurante"].nunique()}')
df.head()

DataFrame final: 1012 reseñas únicas

Restaurantes únicos: 13


,calificacion_resena,texto,usuario,pagina,nombre_restaurante,enlace,calificacion_general,direccion_restaurante
0,5.0,"Delicioso! 100% recomendable. Sopa criolla, ca...",K. E.,1,Opiniones de Siete Sopas Lince. (Restaurante) ...,https://cityperu.com/restaurante/lince/siete-s...,4.3,"804onal GaiAv. Arequipa 2394, LimaD"
1,4.0,"Local amplio, hay cola para entrar en las hora...",B. S.,1,Opiniones de Siete Sopas Lince. (Restaurante) ...,https://cityperu.com/restaurante/lince/siete-s...,4.3,"804onal GaiAv. Arequipa 2394, LimaD"
2,5.0,"Buena atención , lugar impecable. Había la típ...",J. V.,1,Opiniones de Siete Sopas Lince. (Restaurante) ...,https://cityperu.com/restaurante/lince/siete-s...,4.3,"804onal GaiAv. Arequipa 2394, LimaD"
3,5.0,"""Milanesa de Pollo Bien Pobre"" buenísimo! Cómo...",C. L.,1,Opiniones de Siete Sopas Lince. (Restaurante) ...,https://cityperu.com/restaurante/lince/siete-s...,4.3,"804onal GaiAv. Arequipa 2394, LimaD"
4,3.0,Me encantaría darle 5 estrellas pero esta vez ...,L.,1,Opiniones de Siete Sopas Lince. (Restaurante) ...,https://cityperu.com/restaurante/lince/siete-s...,4.3,"804onal GaiAv. Arequipa 2394, LimaD"


In [10]:
df.tail()

,calificacion_resena,texto,usuario,pagina,nombre_restaurante,enlace,calificacion_general,direccion_restaurante
1007,5.0,"Muy buena comida. Los platos criollos, prepara...",A. L.,1,Opiniones de Restaurante Huancahuasi Javier Pr...,https://cityperu.com/restaurante/la-victoria/r...,4.5,"Av. Javier Prado Este 1405, Lima 15034"
1008,5.0,Muy buena la comida excelente atención trato a...,j. l. r. m.,1,Opiniones de Restaurante Huancahuasi Javier Pr...,https://cityperu.com/restaurante/la-victoria/r...,4.5,"Av. Javier Prado Este 1405, Lima 15034"
1009,4.0,"Muy buena experiencia, la comida exquisita, la...",F. C.,1,Opiniones de Restaurante Huancahuasi Javier Pr...,https://cityperu.com/restaurante/la-victoria/r...,4.5,"Av. Javier Prado Este 1405, Lima 15034"
1010,5.0,"Vale la pena esperar si está lleno, la comida ...",M.,1,Opiniones de Restaurante Huancahuasi Javier Pr...,https://cityperu.com/restaurante/la-victoria/r...,4.5,"Av. Javier Prado Este 1405, Lima 15034"
1011,5.0,"Deliciosa comida Peruana, no hay pierde... la ...",L. P.,1,Opiniones de Restaurante Huancahuasi Javier Pr...,https://cityperu.com/restaurante/la-victoria/r...,4.5,"Av. Javier Prado Este 1405, Lima 15034"
